<a href="https://colab.research.google.com/github/helmernet/Helmernet/blob/main/Mapainterativo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install folium pandas

In [5]:
import pandas as pd

# Datos de Euromonitor International (llegadas internacionales) y Time Out
ciudades = [
    {"Ciudad": "Bangkok", "País": "Tailandia", "Visitantes (millones)": 32.4, "Ranking": 1},
    {"Ciudad": "Estambul", "País": "Turquía", "Visitantes (millones)": 23.0, "Ranking": 2},
    {"Ciudad": "Londres", "País": "Reino Unido", "Visitantes (millones)": 21.7, "Ranking": 3},
    {"Ciudad": "Hong Kong", "País": "China", "Visitantes (millones)": 20.5, "Ranking": 4},
    {"Ciudad": "La Meca", "País": "Arabia Saudita", "Visitantes (millones)": 19.3, "Ranking": 5},
    {"Ciudad": "Antalya", "País": "Turquía", "Visitantes (millones)": 19.3, "Ranking": 6},
    {"Ciudad": "Dubái", "País": "Emiratos Árabes Unidos", "Visitantes (millones)": 18.2, "Ranking": 7},
    {"Ciudad": "Macao", "País": "China", "Visitantes (millones)": 18.0, "Ranking": 8},
    {"Ciudad": "París", "País": "Francia", "Visitantes (millones)": 17.4, "Ranking": 9},
    {"Ciudad": "Kuala Lumpur", "País": "Malasia", "Visitantes (millones)": 16.5, "Ranking": 10},
    {"Ciudad": "Nueva York", "País": "EE.UU.", "Visitantes (millones)": 16.0, "Ranking": 11},  # Estimación basada en tendencias :cite[6]
    {"Ciudad": "Tokio", "País": "Japón", "Visitantes (millones)": 13.0, "Ranking": 12},  # Según Euromonitor :cite[8]
    {"Ciudad": "Singapur", "País": "Singapur", "Visitantes (millones)": 12.8, "Ranking": 13},
    {"Ciudad": "Ciudad del Cabo", "País": "Sudáfrica", "Visitantes (millones)": 12.5, "Ranking": 14},  # Time Out :cite[9]
    {"Ciudad": "Barcelona", "País": "España", "Visitantes (millones)": 12.0, "Ranking": 15},  # Euromonitor :cite[8]
    {"Ciudad": "Madrid", "País": "España", "Visitantes (millones)": 11.5, "Ranking": 16},  # Euromonitor :cite[8]
    {"Ciudad": "Roma", "País": "Italia", "Visitantes (millones)": 11.0, "Ranking": 17},
    {"Ciudad": "Ámsterdam", "País": "Países Bajos", "Visitantes (millones)": 10.8, "Ranking": 18},
    {"Ciudad": "Sídney", "País": "Australia", "Visitantes (millones)": 10.5, "Ranking": 19},
    {"Ciudad": "Milán", "País": "Italia", "Visitantes (millones)": 10.0, "Ranking": 20},
]

df = pd.DataFrame(ciudades)

In [8]:
!pip install geopy

In [11]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Inicializar el geocodificador
geolocator = Nominatim(user_agent="ciudades_mas_visitadas")

# Configurar un límite de tasa para evitar errores de sobrecarga
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

In [12]:
# Función para obtener latitud y longitud
def obtener_coordenadas(ciudad, pais):
    try:
        location = geocode(f"{ciudad}, {pais}")
        return (location.latitude, location.longitude)
    except:
        return (None, None)

# Aplicar la función a cada fila del DataFrame
df[["Latitud", "Longitud"]] = df.apply(
    lambda row: pd.Series(obtener_coordenadas(row["Ciudad"], row["País"])),
    axis=1
)

# Mostrar el DataFrame con las coordenadas
print(df)

             Ciudad                    País  Visitantes (millones)  Ranking  \
0           Bangkok               Tailandia                   32.4        1   
1          Estambul                 Turquía                   23.0        2   
2           Londres             Reino Unido                   21.7        3   
3         Hong Kong                   China                   20.5        4   
4           La Meca          Arabia Saudita                   19.3        5   
5           Antalya                 Turquía                   19.3        6   
6             Dubái  Emiratos Árabes Unidos                   18.2        7   
7             Macao                   China                   18.0        8   
8             París                 Francia                   17.4        9   
9      Kuala Lumpur                 Malasia                   16.5       10   
10       Nueva York                  EE.UU.                   16.0       11   
11            Tokio                   Japón         

In [13]:
# Ejemplo de corrección manual
df.loc[df["Ciudad"] == "La Meca", ["Latitud", "Longitud"]] = [21.4225, 39.8262]
df.loc[df["Ciudad"] == "Macao", ["Latitud", "Longitud"]] = [22.1987, 113.5439]

In [14]:
import folium

# Crear mapa centrado en el mundo
mapa = folium.Map(location=[20, 0], zoom_start=2)

# Añadir marcadores con popups
for idx, row in df.iterrows():
    folium.Marker(
        location=[row["Latitud"], row["Longitud"]],
        popup=f"<b>{row['Ciudad']}</b><br>Ranking: {row['Ranking']}<br>Visitantes: {row['Visitantes (millones)']}M",
        icon=folium.Icon(color="blue" if row["Ranking"] <= 10 else "green")
    ).add_to(mapa)

# Guardar y mostrar
mapa.save("mapa_ciudades.html")
mapa

In [15]:
import folium
from branca.element import Template, MacroElement

# Crear mapa
mapa = folium.Map(location=[20, 0], zoom_start=2)

# Función para asignar colores según el rango
def get_color(ranking):
    if 1 <= ranking <= 5:
        return 'red'
    elif 6 <= ranking <= 10:
        return 'blue'
    elif 11 <= ranking <= 15:
        return 'green'
    else:
        return 'purple'

# Añadir marcadores con colores
for idx, row in df.iterrows():
    folium.Marker(
        location=[row["Latitud"], row["Longitud"]],
        popup=f"<b>{row['Ciudad']}</b><br>Ranking: {row['Ranking']}<br>Visitantes: {row['Visitantes (millones)']}M",
        icon=folium.Icon(color=get_color(row['Ranking']), icon='info-sign')
    ).add_to(mapa)

# Crear leyenda personalizada
template = """
{% macro html(this, kwargs) %}
<div style="
    position: fixed;
    bottom: 50px;
    right: 50px;
    width: 150px;
    height: 160px;
    z-index:9999;
    font-size:14px;
    background-color:white;
    padding:10px;
    border-radius:5px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.4);
">
    <p style="margin:0 0 5px;"><strong>Leyenda</strong></p>
    <div style="display: flex; align-items: center; margin:5px 0;">
        <div style="background:red; width:20px; height:20px; border-radius:50%;"></div>
        <span style="margin-left:5px;">Rank 1-5</span>
    </div>
    <div style="display: flex; align-items: center; margin:5px 0;">
        <div style="background:blue; width:20px; height:20px; border-radius:50%;"></div>
        <span style="margin-left:5px;">Rank 6-10</span>
    </div>
    <div style="display: flex; align-items: center; margin:5px 0;">
        <div style="background:green; width:20px; height:20px; border-radius:50%;"></div>
        <span style="margin-left:5px;">Rank 11-15</span>
    </div>
    <div style="display: flex; align-items: center; margin:5px 0;">
        <div style="background:purple; width:20px; height:20px; border-radius:50%;"></div>
        <span style="margin-left:5px;">Rank 16-20</span>
    </div>
</div>
{% endmacro %}
"""

macro = MacroElement()
macro._template = Template(template)
mapa.get_root().add_child(macro)

# Mostrar mapa
mapa.save("mapa_ciudades.html")
mapa